In [ ]:
!pip install -Uqqq sentence-transformers faiss-cpu python-telegram-bot PyPDF2 tqdm requests yandexcloud

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.2/470.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 717.1/717.1 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 104.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/

Идентификатор ключа
ajeh6r2s4md2k4psrdln

Ваш секретный ключ
AQVN0TbFwlmioTPNtreDAC1WOmZU68A6cF6eXuWb

In [ ]:
import os
import PyPDF2
import re
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

DATA_DIR = "sp_data"
INDEX_PATH = "sp_index.faiss"
METADATA_PATH = "metadata.json"

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def extract_text_from_pdf(pdf_path):
    """Извлекает текст из PDF с сохранением структуры"""
    text = ""
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() + "\n"
    return text

In [ ]:
def clean_text(text):
    """Очистка и нормализация текста"""
    text = re.sub(r'\s+', ' ', text)  # Удаление лишних пробелов
    text = re.sub(r'[^\w\s.,;:!?()\-–%№«»§]', '', text)  # Удаление спецсимволов
    text = text.lower()

    return text.strip()

In [ ]:
def chunk_text(text, chunk_size=1000, overlap=100):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
    return chunks

In [ ]:
# Обработка документов
documents = []
embeddings = []

for file in tqdm(os.listdir(DATA_DIR), desc="Обработка документов"):
    if file.endswith(".pdf"):
        file_path = os.path.join(DATA_DIR, file)
        text = extract_text_from_pdf(file_path)
        cleaned_text = clean_text(text)
        chunks = chunk_text(cleaned_text)

        for i, chunk in enumerate(chunks):
            # Сохраняем метаданные
            documents.append({
                "doc_id": f"{file}_{i}",
                "source": file,
                "chunk_index": i,
                "text": chunk
            })

            # Создаем эмбеддинг
            embedding = model.encode([chunk])[0]
            embeddings.append(embedding)

Обработка документов:   3%|▎         | 2/61 [00:00<00:06,  8.63it/s]/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Обработка документов: 100%|██████████| 61/61 [03:21<00:00,  3.31s/it]


In [ ]:
# Создание индекса FAISS
embeddings = np.array(embeddings).astype('float32')
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

In [ ]:
# Сохранение индекса и метаданных
faiss.write_index(index, INDEX_PATH)
with open(METADATA_PATH, "w", encoding="utf-8") as f:
    import json
    json.dump(documents, f, ensure_ascii=False)

print(f"✅ База знаний создана: {len(documents)} чанков")

✅ База знаний создана: 433 чанков


In [ ]:
#Поиск по созданной базе данных
def search(query):
    # 1. Загружаем индекс и метаданные
    index = faiss.read_index("sp_index.faiss")
    with open("metadata.json", encoding="utf-8") as f:
        docs = json.load(f)

    # 2. Ищем похожие документы
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, k=3)

    # 3. Возвращаем тексты
    return [docs[i] for i in indices[0]]

In [ ]:
def process_document(text):
    # 1. Полная очистка без потерь
    cleaned_text = clean_text(text)

    # 2. Чанкирование вместо обрезки
    chunks = chunk_text(cleaned_text,
                       chunk_size=1000,  # ~150 слов
                       overlap=100)      # Контекстное перекрытие

    # 3. Обработка каждого чанка
    for chunk in chunks:
        create_embedding(chunk)

In [ ]:
import requests

url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
headers = {
    "Authorization": "Api-Key AQVN0TbFwlmioTPNtreDAC1WOmZU68A6cF6eXuWb",  # Добавьте префикс "Api-Key"
    "Content-Type": "application/json"
}
data = {
    "modelUri": "gpt://b1gqog5m3dhlf83oafu9/yandexgpt-lite/latest",  # Добавлено /latest
    "messages": [
        {
            "role": "user",
            "text": "Каки требования по кратности воздухообмена в офисах?"
        }
    ],
    "completionOptions": {
        "stream": False,
        "temperature": 0.2,
        "maxTokens": 2000
    }
}

response = requests.post(url, headers=headers, json=data)
print(response.json())

{'result': {'alternatives': [{'message': {'role': 'assistant', 'text': 'Требования по кратности воздухообмена в офисах регламентируются различными нормативными документами, такими как СНиП (Строительные нормы и правила), СанПиН (Санитарные правила и нормы) и другие.\n\nСогласно этим документам, кратность воздухообмена в офисных помещениях должна обеспечивать комфортные условия для работы и соответствовать определённым нормам. Например, в зависимости от количества людей в помещении и его назначения, кратность воздухообмена может варьироваться.\n\nДля точного определения требований по кратности воздухообмена необходимо учитывать:\n* назначение помещения (например, офисное пространство, переговорная комната);\n* количество людей, находящихся в помещении;\n* наличие и количество офисной техники;\n* характеристики помещения (площадь, высота потолков).\n\nРекомендуется обратиться к актуальным версиям нормативных документов или проконсультироваться с профессионалами в области вентиляции и кон